# Hospital Operations & Revenue Risk Intelligence Platform
## Phase 5 — Deployment and API Integration

**Programme:** Advanced Certificate Programme in Applied AI and Machine Learning
**Platform:** Kaggle Notebook

---

### What This Phase Does

Phase 5 bridges the gap between a trained ML model (Phase 3/4) and a real-world usable system that hospital staff, dashboards, or other software can actually call.

**What we built and deployed:**
- A **FastAPI** application serving both trained XGBoost models as REST API endpoints
- Containerized with **Docker** so it runs identically anywhere
- Deployed on **Hugging Face Spaces** (Docker SDK) — giving us a public URL anyone can reach without installing anything

**Architecture:**
```
Hospital System / Dashboard / This Notebook
        |
        | HTTP POST request (JSON)
        v
Hugging Face Spaces (Docker container running Python 3.12)
        |
        | FastAPI receives request --> schemas.py validates input
        | --> preprocessing.py converts raw fields to model features
        | --> XGBoost model predicts probabilities
        | --> logging_utils.py logs prediction (timestamp + feature hash)
        v
JSON response with prediction + probabilities + reliability warning
```

**Live API URL:** `https://yashrajy-hospital-risk-claim-api.hf.space`
**Swagger docs:** `https://yashrajy-hospital-risk-claim-api.hf.space/docs`
**Space repo:** `https://huggingface.co/spaces/Yashrajy/hospital-risk-claim-api`

---


## Step 1 — Import Libraries

Only `requests` (for HTTP calls to the deployed API) and `pandas` are needed here.
The models run inside the Docker container on Hugging Face — not in this notebook.


In [1]:
import requests
import pandas as pd
import json
import time

# Live Hugging Face Spaces URL for our deployed API
# UPDATE this if your Space URL is different
API_BASE_URL = "https://yashrajy-hospital-risk-claim-api.hf.space"

print(f"API base URL : {API_BASE_URL}")
print(f"Health check : {API_BASE_URL}/health")
print(f"Risk predict : {API_BASE_URL}/predict/risk")
print(f"Claim predict: {API_BASE_URL}/predict/claim")
print(f"Swagger docs : {API_BASE_URL}/docs")


API base URL : https://yashrajy-hospital-risk-claim-api.hf.space
Health check : https://yashrajy-hospital-risk-claim-api.hf.space/health
Risk predict : https://yashrajy-hospital-risk-claim-api.hf.space/predict/risk
Claim predict: https://yashrajy-hospital-risk-claim-api.hf.space/predict/claim
Swagger docs : https://yashrajy-hospital-risk-claim-api.hf.space/docs


---
## Step 2 — Deployment Architecture and Source Files

The full deployment package contains the following files, hosted on Hugging Face:

```
hospital-risk-claim-api/               <- Space repository root
├── Dockerfile                         <- Builds the container (Python 3.12-slim, port 7860)
├── README.md                          <- Space description with YAML frontmatter (sdk: docker)
├── requirements.txt                   <- Pinned dependencies (FastAPI, XGBoost, scikit-learn, etc.)
├── deployment_runbook.md              <- Full build / run / operate guide
├── sample_requests.md                 <- Example API calls with real captured outputs
└── app/                               <- FastAPI application package
    ├── __init__.py                    <- Python package marker
    ├── main.py                        <- FastAPI app: endpoints, model loading, lifespan
    ├── schemas.py                     <- Pydantic request/response validation models
    ├── preprocessing.py               <- Raw input -> model feature vector (mirrors Phase 2)
    ├── logging_utils.py               <- Logs timestamp + model version + SHA-256 feature hash
    └── models/                        <- Trained model artifacts (baked into the Docker image)
        ├── risk_model_xgb.joblib
        ├── risk_model_label_encoders.joblib
        ├── claim_model_xgb.joblib
        ├── claim_model_label_encoders.joblib
        └── claim_model_provider_rejection_rate.joblib
```

### API Endpoints

| Method | Endpoint | Purpose |
|---|---|---|
| GET | `/health` | Confirms both models are loaded in the running container |
| POST | `/predict/risk` | Predicts visit risk: Low / Medium / High |
| POST | `/predict/claim` | Predicts claim outcome: Paid / Pending / Rejected |
| GET | `/docs` | Interactive Swagger UI — auto-generated by FastAPI |

### Key Design Decisions

**1. Raw business fields as input, not encoded features**
The API accepts fields a hospital system would naturally have (e.g. `department: "ICU"`, `visit_date: "2025-08-01"`). `preprocessing.py` converts these into the exact encoded feature vector each model expects, replicating Phase 2's feature engineering exactly — including the registration-date clipping fix for the synthetic data quality issue found in Phase 2.

**2. Leakage-free provider rejection rate**
The claim model uses `provider_rejection_rate` as a feature. The API looks this up from `claim_model_provider_rejection_rate.joblib` (training-period-only rates) rather than recomputing it from any live data — this is the same leakage fix applied in Phase 3.

**3. Every prediction is logged**
`logging_utils.py` writes one structured JSON log line per prediction: timestamp, model name, model version, and a SHA-256 hash of the input features. The hash protects patient privacy while still enabling audit trail and drift detection in Phase 6.

**4. Explicit reliability warnings in every response**
Both models have limited predictive power (Phase 3 signal check: max effect size ~0.02, negligible). Every API response includes a `warning` field stating this directly — no consumer of this API can be unaware of the limitations.


---
## Step 3 — Health Check: Confirm the API is Live

`/health` confirms the container is running and both models loaded successfully.
If either model failed to load at container startup, the container would have
crashed immediately — this fail-fast behavior is intentional.


In [2]:
response = requests.get(f"{API_BASE_URL}/health", timeout=30)

print(f"HTTP Status Code : {response.status_code}")
print(f"Response         : {response.text}")
print()

health = response.json()
print(f"  status             : {health['status']}")
print(f"  risk_model_loaded  : {health['risk_model_loaded']}")
print(f"  claim_model_loaded : {health['claim_model_loaded']}")

if response.status_code == 200 and health['status'] == 'ok':
    print("\nAPI is live and both models are loaded successfully.")
else:
    print("\nWARNING: Health check failed. Check the Hugging Face Space logs.")


HTTP Status Code : 200
Response         : {"status":"ok","risk_model_loaded":true,"claim_model_loaded":true}

  status             : ok
  risk_model_loaded  : True
  claim_model_loaded : True

API is live and both models are loaded successfully.


---
## Step 4 — Model A: Visit Risk Prediction (Single Request)

We call `/predict/risk` with a sample hospital visit.

**What the API does internally (preprocessing.py):**
1. Encodes categorical fields using saved LabelEncoders from training
2. Computes `days_since_registration` = visit_date - registration_date (clips at 0)
3. Extracts time features from visit_date (month, day_of_week, is_weekend, quarter)
4. Assembles the exact 15-feature vector in the order XGBoost was trained on
5. Returns predicted class + full probability breakdown


In [3]:
risk_request = {
    "department": "ICU",
    "visit_type": "ICU",
    "city": "Delhi",
    "gender": "M",
    "age": 65,
    "chronic_flag": 1,
    "length_of_stay_hours": 48.0,
    "doctor_id": 101,
    "visit_frequency": 3,
    "avg_los_per_patient": 40.0,
    "registration_date": "2025-01-15",
    "visit_date": "2025-08-01"
}

print("Input:")
print(json.dumps(risk_request, indent=2))
print()

response = requests.post(f"{API_BASE_URL}/predict/risk", json=risk_request, timeout=30)
risk_result = response.json()

print(f"HTTP Status: {response.status_code}")
print()
print("Response:")
print(json.dumps(risk_result, indent=2))


Input:
{
  "department": "ICU",
  "visit_type": "ICU",
  "city": "Delhi",
  "gender": "M",
  "age": 65,
  "chronic_flag": 1,
  "length_of_stay_hours": 48.0,
  "doctor_id": 101,
  "visit_frequency": 3,
  "avg_los_per_patient": 40.0,
  "registration_date": "2025-01-15",
  "visit_date": "2025-08-01"
}

HTTP Status: 200

Response:
{
  "predicted_risk": "Low",
  "probability_low": 0.45205429196357727,
  "probability_medium": 0.3508724868297577,
  "probability_high": 0.19707323610782623,
  "model_version": "xgboost-v1-phase3",
  "warning": "This model's test-set performance (Macro AUROC 0.53) is barely above random. A formal statistical signal check found negligible association between risk_score and all available features. Treat this prediction as decision-support only, not a reliable risk assessment. See risk_model_card.md for details."
}


In [4]:
print("=" * 55)
print("VISIT RISK PREDICTION RESULT")
print("=" * 55)
print(f"Predicted Risk : {risk_result['predicted_risk']}")
print()
print("Probability Breakdown:")
print(f"  Low    : {risk_result['probability_low']:.1%}")
print(f"  Medium : {risk_result['probability_medium']:.1%}")
print(f"  High   : {risk_result['probability_high']:.1%}")
print()
print(f"Model Version  : {risk_result['model_version']}")
print()
print("Reliability Warning:")
print(f"  {risk_result['warning']}")


VISIT RISK PREDICTION RESULT
Predicted Risk : Low

Probability Breakdown:
  Low    : 45.2%
  Medium : 35.1%
  High   : 19.7%

Model Version  : xgboost-v1-phase3

Reliability Warning:
  This model's test-set performance (Macro AUROC 0.53) is barely above random. A formal statistical signal check found negligible association between risk_score and all available features. Treat this prediction as decision-support only, not a reliable risk assessment. See risk_model_card.md for details.


---
## Step 5 — Model B: Claim Outcome Prediction (Single Request)

We call `/predict/claim` with a sample billing record.

Note: fewer fields than the risk endpoint — the claim model deliberately
excludes demographic attributes (age, gender, city) as predictors to
avoid building demographic bias into approval predictions. This design
decision is documented in Phase 3's feature selection section.


In [5]:
claim_request = {
    "insurance_provider": "SecureLife",
    "department": "Cardiology",
    "visit_type": "OPD",
    "billed_amount": 35000.0,
    "length_of_stay_hours": 12.0,
    "chronic_flag": 0,
    "registration_date": "2025-02-01",
    "visit_date": "2025-08-01"
}

print("Input:")
print(json.dumps(claim_request, indent=2))
print()

response = requests.post(f"{API_BASE_URL}/predict/claim", json=claim_request, timeout=30)
claim_result = response.json()

print(f"HTTP Status: {response.status_code}")
print()
print("Response:")
print(json.dumps(claim_result, indent=2))


Input:
{
  "insurance_provider": "SecureLife",
  "department": "Cardiology",
  "visit_type": "OPD",
  "billed_amount": 35000.0,
  "length_of_stay_hours": 12.0,
  "chronic_flag": 0,
  "registration_date": "2025-02-01",
  "visit_date": "2025-08-01"
}

HTTP Status: 200

Response:
{
  "predicted_claim_status": "Paid",
  "probability_paid": 0.669986367225647,
  "probability_pending": 0.22458577156066895,
  "probability_rejected": 0.10542786121368408,
  "model_version": "xgboost-v1-phase3",
  "warning": "This model's test-set performance for the Rejected class (AUROC 0.65) is modest. A formal statistical signal check found only weak association in the underlying data. Treat this prediction as a secondary screening signal, not a substitute for billing team review. See claim_model_card.md for details."
}


In [6]:
print("=" * 55)
print("CLAIM OUTCOME PREDICTION RESULT")
print("=" * 55)
print(f"Predicted Status : {claim_result['predicted_claim_status']}")
print()
print("Probability Breakdown:")
print(f"  Paid     : {claim_result['probability_paid']:.1%}")
print(f"  Pending  : {claim_result['probability_pending']:.1%}")
print(f"  Rejected : {claim_result['probability_rejected']:.1%}")
print()
print(f"Model Version  : {claim_result['model_version']}")
print()
print("Reliability Warning:")
print(f"  {claim_result['warning']}")


CLAIM OUTCOME PREDICTION RESULT
Predicted Status : Paid

Probability Breakdown:
  Paid     : 67.0%
  Pending  : 22.5%
  Rejected : 10.5%

Model Version  : xgboost-v1-phase3

Reliability Warning:
  This model's test-set performance for the Rejected class (AUROC 0.65) is modest. A formal statistical signal check found only weak association in the underlying data. Treat this prediction as a secondary screening signal, not a substitute for billing team review. See claim_model_card.md for details.


---
## Step 6 — Request Schema Validation

Phase 5 requires "request and response schema validation." FastAPI + Pydantic
handles this automatically via `schemas.py`. Invalid requests are rejected
BEFORE reaching the model — the API returns HTTP 422 with a clear error message.

We demonstrate three validation cases. The API is working **correctly** when
it rejects these — a 422 is the expected, desired response.


In [7]:
# Test 1: Invalid department (not in the trained Literal enum)
print("TEST 1 — Invalid department value ('Pediatrics')")
print("-" * 55)

r1 = requests.post(
    f"{API_BASE_URL}/predict/risk",
    json={**risk_request, "department": "Pediatrics"},
    timeout=30
)
print(f"HTTP Status: {r1.status_code}  (expected: 422)")
e1 = r1.json()["detail"][0]
print(f"Error type : {e1['type']}")
print(f"Field      : {e1['loc'][-1]}")
print(f"Message    : {e1['msg']}")
print()

# Test 2: Missing required fields
print("TEST 2 — Missing required fields")
print("-" * 55)

r2 = requests.post(
    f"{API_BASE_URL}/predict/risk",
    json={"department": "ICU", "visit_type": "ICU"},
    timeout=30
)
print(f"HTTP Status: {r2.status_code}  (expected: 422)")
errors_2 = r2.json()["detail"]
print(f"Missing fields detected: {len(errors_2)}")
for e in errors_2[:4]:
    print(f"  Missing: {e['loc'][-1]}")
if len(errors_2) > 4:
    print(f"  ... and {len(errors_2)-4} more")
print()

# Test 3: Out-of-range numeric value
print("TEST 3 — Out-of-range value (age = -5)")
print("-" * 55)

r3 = requests.post(
    f"{API_BASE_URL}/predict/risk",
    json={**risk_request, "age": -5},
    timeout=30
)
print(f"HTTP Status: {r3.status_code}  (expected: 422)")
e3 = r3.json()["detail"][0]
print(f"Error type : {e3['type']}")
print(f"Field      : {e3['loc'][-1]}")
print(f"Message    : {e3['msg']}")


TEST 1 — Invalid department value ('Pediatrics')
-------------------------------------------------------
HTTP Status: 422  (expected: 422)
Error type : literal_error
Field      : department
Message    : Input should be 'Cardiology', 'ER', 'General', 'ICU', 'Neurology' or 'Orthopedics'

TEST 2 — Missing required fields
-------------------------------------------------------
HTTP Status: 422  (expected: 422)
Missing fields detected: 10
  Missing: city
  Missing: gender
  Missing: age
  Missing: chronic_flag
  ... and 6 more

TEST 3 — Out-of-range value (age = -5)
-------------------------------------------------------
HTTP Status: 422  (expected: 422)
Error type : greater_than_equal
Field      : age
Message    : Input should be greater than or equal to 0


---
## Step 7 — Batch Predictions on Real Test Data

We load the actual test records from Phase 3/4 and send them through
the live API. This shows the full pipeline working end-to-end — not just
with hand-crafted example inputs, but with real records from the dataset.


In [12]:
# Load model_table.csv from the Kaggle dataset
data = pd.read_csv("/kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/model_table.csv")
data["visit_date"] = pd.to_datetime(data["visit_date"])
data["registration_date"] = pd.to_datetime(data["registration_date"])

# Reproduce the same time-based test split used in Phase 3
data_sorted = data.sort_values("visit_date").reset_index(drop=True)
test_start_idx = int(len(data_sorted) * 0.80)
test_records = data_sorted.iloc[test_start_idx:test_start_idx+10].copy()

print(f"Total records in model_table  : {len(data):,}")
print(f"Test records (last 20%)       : {len(data_sorted) - test_start_idx:,}")
print(f"Records sent through API now  : 10")
print()
print("Sending 10 test records through the live risk API...")


Total records in model_table  : 25,000
Test records (last 20%)       : 5,000
Records sent through API now  : 10

Sending 10 test records through the live risk API...


In [13]:
risk_batch = []

for _, row in test_records.iterrows():
    payload = {
        "department": row["department"],
        "visit_type": row["visit_type"],
        "city": row["city"],
        "gender": row["gender"],
        "age": int(row["age"]),
        "chronic_flag": int(row["chronic_flag"]),
        "length_of_stay_hours": float(row["length_of_stay_hours"]),
        "doctor_id": int(row["doctor_id"]),
        "visit_frequency": int(row["visit_frequency"]),
        "avg_los_per_patient": float(row["avg_los_per_patient"]),
        "registration_date": str(row["registration_date"].date()),
        "visit_date": str(row["visit_date"].date())
    }

    resp = requests.post(f"{API_BASE_URL}/predict/risk", json=payload, timeout=30)

    if resp.status_code == 200:
        r = resp.json()
        risk_batch.append({
            "visit_id": int(row["visit_id"]),
            "department": row["department"],
            "actual_risk": row["risk_score"],
            "predicted_risk": r["predicted_risk"],
            "prob_low": round(r["probability_low"], 3),
            "prob_medium": round(r["probability_medium"], 3),
            "prob_high": round(r["probability_high"], 3),
            "correct": row["risk_score"] == r["predicted_risk"]
        })

    time.sleep(0.3)   # small delay — polite to the free-tier API

risk_batch_df = pd.DataFrame(risk_batch)
print("Risk Model — Batch Prediction Results (10 records):")
display(risk_batch_df)
print(f"\nCorrect: {risk_batch_df['correct'].sum()} / {len(risk_batch_df)}")
print("(Modest accuracy is expected — consistent with Phase 4's AUROC ~0.53 finding)")


Risk Model — Batch Prediction Results (10 records):


,visit_id,department,actual_risk,predicted_risk,prob_low,prob_medium,prob_high,correct
0,9322,ICU,Low,Low,0.518,0.286,0.196,True
1,23935,ICU,Low,Low,0.548,0.270,0.182,True
2,6610,General,Medium,Low,0.398,0.343,0.258,False
3,3886,Cardiology,Low,Low,0.493,0.333,0.174,True
4,17280,General,High,Low,0.478,0.338,0.184,False
5,13623,Orthopedics,High,Low,0.517,0.280,0.204,False
6,18459,Neurology,Medium,Low,0.548,0.274,0.178,False
7,19879,General,High,Low,0.564,0.258,0.177,False
8,22695,ER,Low,Low,0.531,0.281,0.189,True
9,19875,ER,Low,Low,0.537,0.269,0.194,True



Correct: 5 / 10
(Modest accuracy is expected — consistent with Phase 4's AUROC ~0.53 finding)


In [14]:
claim_batch = []

print("Sending same 10 records through the live claim API...")
print()

for _, row in test_records.iterrows():
    payload = {
        "insurance_provider": row["insurance_provider"],
        "department": row["department"],
        "visit_type": row["visit_type"],
        "billed_amount": float(row["billed_amount"]),
        "length_of_stay_hours": float(row["length_of_stay_hours"]),
        "chronic_flag": int(row["chronic_flag"]),
        "registration_date": str(row["registration_date"].date()),
        "visit_date": str(row["visit_date"].date())
    }

    resp = requests.post(f"{API_BASE_URL}/predict/claim", json=payload, timeout=30)

    if resp.status_code == 200:
        r = resp.json()
        claim_batch.append({
            "visit_id": int(row["visit_id"]),
            "insurance_provider": row["insurance_provider"],
            "billed_amount": row["billed_amount"],
            "actual_status": row["claim_status"],
            "predicted_status": r["predicted_claim_status"],
            "prob_paid": round(r["probability_paid"], 3),
            "prob_pending": round(r["probability_pending"], 3),
            "prob_rejected": round(r["probability_rejected"], 3),
            "correct": row["claim_status"] == r["predicted_claim_status"]
        })

    time.sleep(0.3)

claim_batch_df = pd.DataFrame(claim_batch)
print("Claim Model — Batch Prediction Results (10 records):")
display(claim_batch_df)
print(f"\nCorrect: {claim_batch_df['correct'].sum()} / {len(claim_batch_df)}")


Sending same 10 records through the live claim API...

Claim Model — Batch Prediction Results (10 records):


,visit_id,insurance_provider,billed_amount,actual_status,predicted_status,prob_paid,prob_pending,prob_rejected,correct
0,9322,HealthPlus,7618.69,Paid,Paid,0.555,0.316,0.129,True
1,23935,MediCareX,9791.68,Paid,Paid,0.664,0.183,0.153,True
2,6610,CareOne,17664.93,Paid,Paid,0.433,0.274,0.293,True
3,3886,HealthPlus,4891.76,Pending,Paid,0.606,0.263,0.131,False
4,17280,MediCareX,895.24,Paid,Paid,0.715,0.225,0.060,True
5,13623,CareOne,11385.48,Paid,Paid,0.660,0.211,0.129,True
6,18459,HealthPlus,34678.32,Pending,Paid,0.631,0.264,0.105,False
7,19879,SecureLife,18993.73,Pending,Paid,0.592,0.151,0.257,False
8,22695,MediCareX,38795.66,Paid,Paid,0.644,0.262,0.095,True
9,19875,SecureLife,4311.10,Pending,Paid,0.671,0.273,0.055,False



Correct: 6 / 10


---
## Phase 5 Summary

| Deliverable | Details | Status |
|---|---|---|
| FastAPI service | `/health`, `/predict/risk`, `/predict/claim` | Done |
| Request validation | Pydantic schema — 422 on invalid input | Done |
| Response validation | Structured JSON with prediction + probabilities + warning | Done |
| Prediction logging | Timestamp + model version + SHA-256 feature hash | Done |
| Docker container | `python:3.12-slim`, port 7860, non-root user, HEALTHCHECK | Done |
| Deployment | Hugging Face Spaces (Docker SDK), public URL | Done |
| Live API verification | Health check, predictions, validation errors all demonstrated | Done |
| Batch integration | 10 real test records from Phase 3/4 sent through live API | Done |
| Runbook | `deployment_runbook.md` — build/run/operate guide | Done |
| Sample requests | `sample_requests.md` — real captured API responses | Done |

### Live Deployment
- **Space:** https://huggingface.co/spaces/Yashrajy/hospital-risk-claim-api
- **API:** https://yashrajy-hospital-risk-claim-api.hf.space
- **Docs:** https://yashrajy-hospital-risk-claim-api.hf.space/docs

### Next: Phase 6 — Monitoring, Drift Detection, and Governance
